In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import numpy as np
import re
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

from sentence_transformers import SentenceTransformer
def semantic_consistency(dialog: str):
    lines = [x.strip() for x in dialog.split("\n") if x.strip()]

    if len(lines) < 2:
        return 0.0

    embeds = model.encode(lines, convert_to_tensor=True)

    similarities = []
    for i in range(len(embeds) - 1):
        emb_i_np = embeds[i].cpu().numpy()
        emb_i_plus_1_np = embeds[i+1].cpu().numpy()

        sim = float(np.dot(emb_i_np, emb_i_plus_1_np) /
                    (np.linalg.norm(emb_i_np)*np.linalg.norm(emb_i_plus_1_np)))
        similarities.append(sim)

    return float(np.mean(similarities))

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def prompt_output_alignment(prompt, output):
    emb_prompt = model.encode(prompt, convert_to_tensor=True)
    emb_output = model.encode(output, convert_to_tensor=True)
    score = float(util.cos_sim(emb_prompt, emb_output)[0][0])
    return score

## Compare between base model and fine-tune model

In [ ]:
import pandas as pd

base_model_output = pd.read_csv("evaluation_results_base.csv")['base_model_output']

base_model = []
for dialog in base_model_output:
    score = semantic_consistency(dialog)
    base_model.append({"dialog": dialog, "score": score})
avg_score = np.mean([item['score'] for item in base_model])
print(f"Average Semantic Consistency Score for Base Model: {avg_score}")

base_model = []
for dialog in base_model_output:
    score = prompt_output_alignment("Please summarize the following conversation.", dialog)
    base_model.append({"dialog": dialog, "score": score})
avg_score = np.mean([item['score'] for item in base_model])
print(f"Average Prompt-Output Alignment Score for Base Model: {avg_score}")

Average Semantic Consistency Score for Base Model: 0.41743566522830433
Average Prompt-Output Alignment Score for Base Model: 0.30099733769893644


In [ ]:
lora_model_output = pd.read_csv("evaluation_results_lora.csv")['lora_model_output']

lora_model = []
for dialog in lora_model_output:
    score = semantic_consistency(dialog)
    lora_model.append({"dialog": dialog, "score": score})
avg_lora_score = np.mean([item['score'] for item in lora_model])
print(f"Average Semantic Consistency Score for LoRA Model: {avg_lora_score}")

lora_model = []
for dialog in lora_model_output:
    score = prompt_output_alignment("Please summarize the following conversation.", dialog)
    lora_model.append({"dialog": dialog, "score": score})
avg_lora_score = np.mean([item['score'] for item in lora_model])
print(f"Average Prompt-Output Alignment Score for LoRA Model: {avg_lora_score}")

Average Semantic Consistency Score for LoRA Model: 0.5810534539745441
Average Prompt-Output Alignment Score for LoRA Model: 0.3855435025691986


## Compare fine-tune model with fine-tune+RAG

In [ ]:
lora_model_output = pd.read_csv("rag_evaluation_results.csv")['lora_model_output']

lora_model = []
for dialog in lora_model_output:
    score = semantic_consistency(dialog)
    lora_model.append({"dialog": dialog, "score": score})
avg_lora_score = np.mean([item['score'] for item in lora_model])
print(f"Average Semantic Consistency Score for LoRA Model: {avg_lora_score}")

lora_model = []
for dialog in lora_model_output:
    score = prompt_output_alignment("Please summarize the following conversation.", dialog)
    lora_model.append({"dialog": dialog, "score": score})
avg_lora_score = np.mean([item['score'] for item in lora_model])
print(f"Average Prompt-Output Alignment Score for LoRA Model: {avg_lora_score}")

Average Semantic Consistency Score for LoRA Model: 0.6302865895132224
Average Prompt-Output Alignment Score for LoRA Model: 0.2675786316394806


In [ ]:
lora_model_output = pd.read_csv("rag_evaluation_results_rag.csv")['lora_model_output']

lora_model = []
for dialog in lora_model_output:
    score = semantic_consistency(dialog)
    lora_model.append({"dialog": dialog, "score": score})
avg_lora_score = np.mean([item['score'] for item in lora_model])
print(f"Average Semantic Consistency Score for LoRA Model+RAG: {avg_lora_score}")

lora_model = []
for dialog in lora_model_output:
    score = prompt_output_alignment("Please summarize the following conversation.", dialog)
    lora_model.append({"dialog": dialog, "score": score})
avg_lora_score = np.mean([item['score'] for item in lora_model])
print(f"Average Prompt-Output Alignment Score for LoRA Model+RAG: {avg_lora_score}")

Average Semantic Consistency Score for LoRA Model+RAG: 0.5386222743866396
Average Prompt-Output Alignment Score for LoRA Model+RAG: 0.24965313289846694
